# ASTRA ARC-AGI-3 — Emergency official starter, CPU fallback

Source: ARC-AGI-3-Kaggle-Starter (arcprize/ARC-AGI-3-Kaggle-Starter). Baseline agent: MyAgent with random decisions.

Commit stage creates a placeholder submission.parquet with the exact public starter schema. This file is not an official game result. In hidden competition rerun, MyAgent calls the official gateway: Kaggle creates the real file after actions. The placeholder is not written in rerun mode. This is a separate emergency baseline, not a claim of a competitive score.


In [ ]:
# Kaggle-safe source-preserving commit cell
exec(compile("import os\nfrom pathlib import Path\n\nTRUE_RERUN = bool(os.getenv(\"KAGGLE_IS_COMPETITION_RERUN\"))\nprint(\"ASTRA ARC-AGI-3 stage:\", \"REAL_RERUN\" if TRUE_RERUN else \"COMMIT_PREFLIGHT\")\nif not TRUE_RERUN:\n    import pandas as pd\n    output = Path(\"/kaggle/working/submission.parquet\")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(\n        data=[[\"1_0\", \"1\", True, 1]],\n        columns=[\"row_id\", \"game_id\", \"end_of_game\", \"score\"],\n    ).to_parquet(output, index=False)\n    assert output.is_file() and output.stat().st_size > 0\n    print(\"ASTRA_COMMIT_SUBMISSION_FILE: PASS\", output, output.stat().st_size)\n", "astra_arc3_commit_parquet", "exec"), globals())


In [ ]:
# Kaggle-safe embedded Python body; preserves indentation during notebook imports
exec(compile("# Only execute the official framework during the hidden competition sidecar.\nif TRUE_RERUN:\n    import os\n    import sys\n    import shutil\n    import subprocess\n    import time\n    from pathlib import Path\n    from urllib.request import urlopen\n\n    source = Path(\"/kaggle/input/competitions/arc-prize-2026-arc-agi-3\")\n    wheels = source / \"arc_agi_3_wheels\"\n    framework = source / \"ARC-AGI-3-Agents\"\n    assert wheels.is_dir(), \"ARC-AGI-3 competition wheelhouse missing\"\n    assert framework.is_dir(), \"ARC-AGI-3-Agents official framework missing\"\n\n    subprocess.check_call([\n        sys.executable, \"-m\", \"pip\", \"install\", \"--no-index\", \"--disable-pip-version-check\",\n        \"--find-links\", str(wheels), \"arc-agi\", \"python-dotenv\"\n    ])\n\n    agent_src = \"\\\"\\\"\\\"Your ARC-AGI-3 agent. This is the *only* file you should normally edit.\\n\\n`scripts/build_notebook.py` splices the contents of this file into the\\nKaggle submission notebook, so your local dev loop and your Kaggle\\nsubmission stay in lock-step:\\n\\n    [edit my_agent.py] → [make play-local] → [make submit]\\n\\nThe default body below is a port of the Stochastic Goose / random_agent\\nsample — a known-good baseline that produces a valid submission and\\nproves your end-to-end pipeline works. Replace `choose_action` with your\\nreal strategy.\\n\\nContract (enforced by the ARC-AGI-3-Agents framework):\\n  - Subclass `agents.agent.Agent`.\\n  - Class must be named `MyAgent` (the notebook's __init__.py registers it).\\n  - Implement `is_done(frames, latest_frame) -> bool`.\\n  - Implement `choose_action(frames, latest_frame) -> GameAction`.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport random\\nimport time\\nfrom typing import Any\\n\\nfrom arcengine import FrameData, GameAction, GameState\\n\\n# When run inside the ARC-AGI-3-Agents framework (locally or on Kaggle)\\n# the `agents` package is on sys.path, so this import resolves.\\nfrom agents.agent import Agent\\n\\n\\nclass MyAgent(Agent):\\n    \\\"\\\"\\\"Picks legal actions uniformly at random. Replace with your strategy.\\\"\\\"\\\"\\n\\n    # Upper bound on actions per game; the framework also enforces global limits.\\n    MAX_ACTIONS = 80\\n\\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\\n        super().__init__(*args, **kwargs)\\n        # Seed per game_id so replays from the same game are reproducible but\\n        # different games explore independently.\\n        seed = int(time.time() * 1_000_000) + hash(self.game_id) % 1_000_000\\n        random.seed(seed)\\n\\n    @property\\n    def name(self) -> str:\\n        return f\\\"{super().name}.{self.MAX_ACTIONS}\\\"\\n\\n    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:\\n        # Stop once we win. Don't stop on GAME_OVER — we want to RESET and retry.\\n        return latest_frame.state is GameState.WIN\\n\\n    def choose_action(\\n        self, frames: list[FrameData], latest_frame: FrameData\\n    ) -> GameAction:\\n        # First call or after a death → reset the level.\\n        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):\\n            return GameAction.RESET\\n\\n        # ── Per-game strategy fork ───────────────────────────────────────────\\n        # By default every game uses the same uniformly-random strategy in the\\n        # `else` branch below. This `if` shows ONE example of giving a single\\n        # game its own heuristic: on LS20 we bias the random pick so ACTION4\\n        # is twice as likely as any other action. Add more `elif` branches to\\n        # specialize other games.\\n        #\\n        # `self.game_id` is set by the framework. It may be the short id\\n        # (\\\"ls20\\\") or include a version suffix (\\\"ls20-9607627b\\\"), so we\\n        # compare on the prefix to be safe.\\n        candidate_actions = [a for a in GameAction if a is not GameAction.RESET]\\n        if self.game_id.split(\\\"-\\\")[0] == \\\"ls20\\\":\\n            weights = [2 if a is GameAction.ACTION4 else 1 for a in candidate_actions]\\n            action = random.choices(candidate_actions, weights=weights, k=1)[0]\\n        else:\\n            action = random.choice(candidate_actions)\\n        # ────────────────────────────────────────────────────────────────────\\n\\n        if action.is_complex():\\n            # ACTION6 takes (x, y) coordinates on a 64×64 grid.\\n            action.set_data(\\n                {\\\"x\\\": random.randint(0, 63), \\\"y\\\": random.randint(0, 63)}\\n            )\\n            action.reasoning = {\\\"why\\\": \\\"random complex action\\\"}\\n        else:\\n            action.reasoning = f\\\"random simple action: {action.value}\\\"\\n        return action\\n\"\n    agent_tmp = Path(\"/tmp/my_agent.py\")\n    agent_tmp.write_text(agent_src, encoding=\"utf-8\")\n\n    target = Path(\"/kaggle/working/ARC-AGI-3-Agents\")\n    shutil.copytree(framework, target, dirs_exist_ok=True)\n    template_dir = target / \"agents\" / \"templates\"\n    template_dir.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(agent_tmp, template_dir / \"my_agent.py\")\n\n    (target / \"agents\" / \"__init__.py\").write_text(\"\"\"from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\nload_dotenv()\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n    'random': Random,\n    'myagent': MyAgent,\n}\n\"\"\", encoding=\"utf-8\")\n    (target / \".env\").write_text(\"\"\"SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=/kaggle/working/server_recording\n\"\"\", encoding=\"utf-8\")\n\n    gateway = \"http://gateway:8001/api/games\"\n    ready = False\n    for attempt in range(120):\n        try:\n            with urlopen(gateway, timeout=3) as response:\n                ready = (response.status == 200)\n            if ready:\n                break\n        except Exception:\n            time.sleep(5)\n    if not ready:\n        raise RuntimeError(\"ARC-AGI-3 gateway never became ready\")\n\n    runtime_env = os.environ.copy()\n    runtime_env[\"MPLBACKEND\"] = \"agg\"\n    subprocess.run([sys.executable, \"main.py\", \"--agent\", \"myagent\"],\n                   cwd=target, env=runtime_env, check=True)\n    print(\"ASTRA_ARC3_GAME_RUN: complete; Kaggle gateway owns submission.parquet\")\n", "astra_arc3_official_agent", "exec"), globals())
